# Setup

In [ ]:
!git clone ANONYMIZED_REPO_URL (I will make this public this later)

In [ ]:
%cd RL_Signaling

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import multiprocessing
import random
from datetime import datetime

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from skopt import Optimizer
from skopt.space import Categorical, Integer, Real
from tqdm import tqdm

from rl_signaling.agents import QLearningAgent, TDLearningAgent, UrnAgent
from rl_signaling.env import NetMultiAgentEnv, TempNetMultiAgentEnv
from rl_signaling.games import create_random_canonical_game, create_random_game
from rl_signaling.simulation import simulation_function, temp_simulation_function


In [ ]:
# Decide where to put the files and do the working
from google.colab import drive
drive.mount('/content/drive')

dump_path = '/content/drive/My Drive/Colab Projects/Python ABMs/Communication/Plots and Datasets/Legacy/'
print("Current Directory:", dump_path)

# Canonical Model

- World States: Two binary variables X, Y
- agents_observed_variables = {0:[0],1:[1]}
- Random Canonical Games
- n_features = 2
- n_signaling_actions = 2
- n_final_actions = 4

## Q-Learning

### Function

In [ ]:
def bayesian_q_parameter_search(
    param_ranges,
    n_calls=30,
    n_trials=10,
    n_episodes=5000,
    base_seed=42,
    n_initial_points=100,
    n_jobs=-1,
    dump_path=dump_path
):
    available_cores = multiprocessing.cpu_count()
    if n_jobs == -1:
        n_jobs = available_cores

    # Build named parameter space
    space = []
    for k, v in param_ranges.items():
        if isinstance(v, Categorical):
            space.append(Categorical(v.categories, name=k))
        elif isinstance(v, Real):
            space.append(Real(v.low, v.high, prior=v.prior, name=k))
        elif isinstance(v, tuple):
            space.append(Real(*v, name=k))
        else:
            raise ValueError(f"Unsupported parameter type for key '{k}': {v}")

    optimizer = Optimizer(
        dimensions=space,
        base_estimator="GP",
        acq_func="EI",
        random_state=base_seed,
    )

    results = []
    all_params = []
    all_nmis = []
    seen_points = set()

    def point_to_hashable(x):
        return tuple(str(xi) for xi in x)

    def single_trial(params_dict, trial_seed):
        np.random.seed(trial_seed)
        random.seed(trial_seed)

        G = nx.DiGraph()
        G.add_nodes_from([0, 1])
        G.add_edges_from([(0, 1), (1, 0)])

        n_agents = 2
        n_features = 2
        n_signaling_actions = 2
        n_final_actions = 4
        agents_observed_variables = {0: [0], 1: [1]}

        game = {
            i: create_random_canonical_game(n_features, n_final_actions, n=1, m=0)
            for i in range(n_agents)
        }

        env = NetMultiAgentEnv(
            n_agents=n_agents,
            n_features=n_features,
            n_signaling_actions=n_signaling_actions,
            n_final_actions=n_final_actions,
            full_information=False,
            game_dicts=game,
            observed_variables=agents_observed_variables,
            agent_type=QLearningAgent,
            initialize=False,
            graph=G
        )

        env.agents = [
            QLearningAgent(
                n_signaling_actions=n_signaling_actions,
                n_final_actions=n_final_actions,
                exploration_rate=params_dict['exploration_rate'],
                exploration_decay=params_dict['exploration_decay'],
                min_exploration_rate=1e-10,
                choice=str(params_dict['choice'])  # normalize category value
            ) for _ in range(n_agents)
        ]

        _, rewards_history, signal_information_history, _, _ = simulation_function(
            n_agents=n_agents,
            n_features=n_features,
            n_signaling_actions=n_signaling_actions,
            n_final_actions=n_final_actions,
            n_episodes=n_episodes,
            with_signals=True,
            plot=False,
            env=env,
            verbose=False
        )

        final_nmi = [
            np.mean(agent_nmi[-n_episodes // 10:]) if len(agent_nmi) >= n_episodes // 10 else 0.0
            for agent_nmi in signal_information_history
        ]
        final_rewards = [
            np.mean(agent_rewards[-n_episodes // 10:]) if len(agent_rewards) >= n_episodes // 10 else 0.0
            for agent_rewards in rewards_history
        ]

        return np.mean(final_nmi), np.std(final_nmi), np.mean(final_rewards), np.std(final_rewards)

    def evaluate_params(params_dict, seed):
        seeds = [seed + i * 1000 for i in range(n_trials)]
        results = Parallel(n_jobs=n_jobs)(delayed(single_trial)(params_dict, s) for s in seeds)
        nmis, nmi_stds, rewards, reward_stds = zip(*results)
        return np.mean(nmis), np.std(nmis), np.mean(rewards), np.std(rewards)

    for _ in tqdm(range(n_initial_points), desc="Initial Exploration"):
        x = optimizer.ask()
        point_hash = point_to_hashable(x)
        if point_hash in seen_points:
            continue
        seen_points.add(point_hash)

        params_dict = dict(zip(param_ranges.keys(), x))
        seed = base_seed + len(results)
        mean_nmi, std_nmi, mean_reward, std_reward = evaluate_params(params_dict, seed)

        optimizer.tell(x, -mean_nmi)
        results.append({
            "params": params_dict,
            "mean_final_nmi": mean_nmi,
            "std_final_nmi": std_nmi,
            "mean_reward": mean_reward,
            "std_reward": std_reward
        })
        all_params.append(x)
        all_nmis.append(mean_nmi)

    for _ in tqdm(range(n_calls - n_initial_points), desc="Bayesian Optimization"):
        x = optimizer.ask()
        point_hash = point_to_hashable(x)
        if point_hash in seen_points:
            continue
        seen_points.add(point_hash)

        params_dict = dict(zip(param_ranges.keys(), x))
        seed = base_seed + len(results)
        mean_nmi, std_nmi, mean_reward, std_reward = evaluate_params(params_dict, seed)

        optimizer.tell(x, -mean_nmi)
        results.append({
            "params": params_dict,
            "mean_final_nmi": mean_nmi,
            "std_final_nmi": std_nmi,
            "mean_reward": mean_reward,
            "std_reward": std_reward
        })
        all_params.append(x)
        all_nmis.append(mean_nmi)

    df = pd.DataFrame([
        {**r['params'],
         'mean_final_nmi': r['mean_final_nmi'],
         'std_final_nmi': r['std_final_nmi'],
         'mean_reward': r['mean_reward'],
         'std_reward': r['std_reward']} for r in results
    ])

    now = datetime.now()
    date_time_str = now.strftime("%Y-%m-%d_%H-%M-%S")
    save_path = dump_path + f"q_bayes_nmi_results_{n_trials}_{date_time_str}.csv"
    df.to_csv(save_path, index=False)
    print(f"Results saved to {save_path}")

    best_idx = np.argmax(all_nmis)
    best_params = dict(zip(param_ranges.keys(), all_params[best_idx]))
    print(f"Best Parameters: {best_params} with NMI = {all_nmis[best_idx]:.4f}, Reward = {results[best_idx]['mean_reward']:.4f}")

    nmis = df['mean_final_nmi']
    rewards = df['mean_reward']
    nmi_errs = df['std_final_nmi'] / np.sqrt(n_trials)
    reward_errs = df['std_reward'] / np.sqrt(n_trials)

    sorted_indices = np.argsort(-nmis)
    pareto_front = []
    max_y = -float('inf')
    for i in sorted_indices:
        x, y = nmis[i], rewards[i]
        if y > max_y:
            pareto_front.append(i)
            max_y = y

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.set_axisbelow(True)
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
    ax.scatter(nmis, rewards, alpha=0.2, color='gray', label='All Trials')

    pareto_x = nmis.iloc[pareto_front]
    pareto_y = rewards.iloc[pareto_front]
    pareto_xerr = nmi_errs.iloc[pareto_front]
    pareto_yerr = reward_errs.iloc[pareto_front]

    ax.errorbar(pareto_x, pareto_y, xerr=pareto_xerr, yerr=pareto_yerr,
                fmt='o', color='red', ecolor='darkred', capsize=4, label='Pareto Frontier')
    ax.plot(pareto_x, pareto_y, color='darkred', linewidth=2)
    ax.set_xlabel("Mean Final NMI")
    ax.set_ylabel("Mean Reward")
    ax.set_title("Pareto Frontier: NMI vs Reward")
    ax.legend()
    plt.tight_layout()

    plot_filename = dump_path + f"q_pareto_frontier_{date_time_str}.png"
    plt.savefig(plot_filename, dpi=300)
    print(f"Pareto plot saved to {plot_filename}")
    plt.show()

    print("\nPareto Frontier Parameter Settings:")
    for i in pareto_front:
        print(f"Params: {results[i]['params']}, "
              f"NMI: {results[i]['mean_final_nmi']:.4f} ± {results[i]['std_final_nmi']:.4f}, "
              f"Reward: {results[i]['mean_reward']:.4f} ± {results[i]['std_reward']:.4f}")

    return results, best_params


### Processing

In [ ]:
# Example usage
param_ranges = {
    "exploration_rate": (0.1, 1),
    "exploration_decay": (0.1, 0.9999999),
    "choice": Categorical(["egreedy", "softmax", "ucb"])
}

q_bayes_results, q_best_params = bayesian_q_parameter_search(
    param_ranges,
    n_calls=200,
    n_trials=100,
    n_episodes=10000,
    n_initial_points=100,
    n_jobs=-1,
    dump_path=dump_path)

## TD-Learning

### Function

In [ ]:
def bayesian_td_parameter_search(
    param_ranges,
    n_calls=30,
    n_trials=10,
    n_episodes=5000,
    base_seed=42,
    n_initial_points=10,
    n_jobs=-1,
    dump_path=dump_path
):
    available_cores = multiprocessing.cpu_count()
    if n_jobs == -1:
        n_jobs = available_cores

    # Remove 'learning_rate' from parameter search (fixed internally)
    param_ranges_wo_lr = {k: v for k, v in param_ranges.items() if k != 'learning_rate'}

    # Build proper skopt parameter space with names
    space = []
    for k, v in param_ranges_wo_lr.items():
        if isinstance(v, Categorical):
            space.append(Categorical(v.categories, name=k))
        elif isinstance(v, Real):
            space.append(Real(v.low, v.high, prior=v.prior, name=k))
        elif isinstance(v, tuple):
            if isinstance(v[0], float):
                space.append(Real(*v, name=k))
            elif isinstance(v[0], int):
                space.append(Integer(*v, name=k))
            else:
                raise ValueError(f"Unsupported tuple value type for key '{k}': {type(v[0])}")
        else:
            raise ValueError(f"Unsupported parameter type for '{k}': {type(v)}")

    optimizer = Optimizer(
        dimensions=space,
        base_estimator="GP",
        acq_func="EI",
        random_state=base_seed,
    )

    results = []
    all_params = []
    all_nmis = []

    def single_trial(params_dict, trial_seed):
        np.random.seed(trial_seed)
        random.seed(trial_seed)

        G = nx.DiGraph()
        G.add_nodes_from([0, 1])
        G.add_edges_from([(0, 1), (1, 0)])

        n_agents = 2
        n_features = 2
        n_signaling_actions = 2
        n_final_actions = 4
        agents_observed_variables = {0: [0], 1: [1]}

        game = {
            i: create_random_canonical_game(n_features, n_final_actions, n=1, m=0)
            for i in range(n_agents)
        }

        env = TempNetMultiAgentEnv(
            n_agents=n_agents,
            n_features=n_features,
            n_signaling_actions=n_signaling_actions,
            n_final_actions=n_final_actions,
            full_information=False,
            game_dicts=game,
            observed_variables=agents_observed_variables,
            agent_type=TDLearningAgent,
            graph=G
        )

        env.agents = [
            TDLearningAgent(
                n_actions=env.max_actions,
                learning_rate=0.1,  # Fixed
                exploration_rate=params_dict['exploration_rate'],
                exploration_decay=params_dict['exploration_decay'],
                min_exploration_rate=1e-10,
                gamma=params_dict['gamma'],
                choice=str(params_dict["choice"])
            ) for _ in range(n_agents)
        ]

        _, rewards_history, signal_information_history, _, _ = temp_simulation_function(
            n_agents=n_agents,
            n_features=n_features,
            n_signaling_actions=n_signaling_actions,
            n_final_actions=n_final_actions,
            n_episodes=n_episodes,
            with_signals=True,
            plot=False,
            env=env,
            verbose=False
        )

        final_nmi = [
            np.mean(agent_nmi[-n_episodes // 10:]) if len(agent_nmi) >= n_episodes // 10 else 0.0
            for agent_nmi in signal_information_history
        ]
        final_rewards = [
            np.mean(agent_rewards[-n_episodes // 10:]) if len(agent_rewards) >= n_episodes // 10 else 0.0
            for agent_rewards in rewards_history
        ]

        return np.mean(final_nmi), np.std(final_nmi), np.mean(final_rewards), np.std(final_rewards)

    def evaluate_params(params_dict, seed):
        seeds = [seed + i * 1000 for i in range(n_trials)]
        results = Parallel(n_jobs=n_jobs)(delayed(single_trial)(params_dict, s) for s in seeds)
        nmis, nmi_stds, rewards, reward_stds = zip(*results)
        return np.mean(nmis), np.std(nmis), np.mean(rewards), np.std(rewards)

    for _ in tqdm(range(n_initial_points), desc="Initial Exploration"):
        x = optimizer.ask()
        params_dict = dict(zip(param_ranges_wo_lr.keys(), x))
        seed = base_seed + len(results)
        mean_nmi, std_nmi, mean_reward, std_reward = evaluate_params(params_dict, seed)
        optimizer.tell(x, -mean_nmi)
        results.append({
            "params": params_dict,
            "mean_final_nmi": mean_nmi,
            "std_final_nmi": std_nmi,
            "mean_reward": mean_reward,
            "std_reward": std_reward
        })
        all_params.append(x)
        all_nmis.append(mean_nmi)

    for _ in tqdm(range(n_calls - n_initial_points), desc="Bayesian Optimization"):
        x = optimizer.ask()
        params_dict = dict(zip(param_ranges_wo_lr.keys(), x))
        seed = base_seed + len(results)
        mean_nmi, std_nmi, mean_reward, std_reward = evaluate_params(params_dict, seed)
        optimizer.tell(x, -mean_nmi)
        results.append({
            "params": params_dict,
            "mean_final_nmi": mean_nmi,
            "std_final_nmi": std_nmi,
            "mean_reward": mean_reward,
            "std_reward": std_reward
        })
        all_params.append(x)
        all_nmis.append(mean_nmi)

    df = pd.DataFrame([{
        **r['params'],
        'mean_final_nmi': r['mean_final_nmi'],
        'std_final_nmi': r['std_final_nmi'],
        'mean_reward': r['mean_reward'],
        'std_reward': r['std_reward']
    } for r in results])

    now = datetime.now()
    date_time_str = now.strftime("%Y-%m-%d_%H-%M-%S")
    save_path = dump_path + f"td_bayes_nmi_results_{n_trials}_{date_time_str}.csv"
    df.to_csv(save_path, index=False)
    print(f"Results saved to {save_path}")

    best_idx = np.argmax(all_nmis)
    best_params = dict(zip(param_ranges_wo_lr.keys(), all_params[best_idx]))
    print(f"Best Parameters: {best_params} with NMI = {all_nmis[best_idx]:.4f}, Reward = {results[best_idx]['mean_reward']:.4f}")

    nmis = df['mean_final_nmi']
    rewards = df['mean_reward']
    nmi_errs = df['std_final_nmi'] / np.sqrt(n_trials)
    reward_errs = df['std_reward'] / np.sqrt(n_trials)

    sorted_indices = np.argsort(-nmis)
    pareto_front = []
    max_y = -float('inf')
    for i in sorted_indices:
        x, y = nmis[i], rewards[i]
        if y > max_y:
            pareto_front.append(i)
            max_y = y

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.set_axisbelow(True)
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
    ax.scatter(nmis, rewards, alpha=0.2, color='gray', label='All Trials')

    pareto_x = nmis.iloc[pareto_front]
    pareto_y = rewards.iloc[pareto_front]
    pareto_xerr = nmi_errs.iloc[pareto_front]
    pareto_yerr = reward_errs.iloc[pareto_front]

    ax.errorbar(pareto_x, pareto_y, xerr=pareto_xerr, yerr=pareto_yerr,
                fmt='o', color='red', ecolor='darkred', capsize=4, label='Pareto Frontier')
    ax.plot(pareto_x, pareto_y, color='darkred', linewidth=2)

    ax.set_xlabel("Mean Final NMI")
    ax.set_ylabel("Mean Reward")
    ax.set_title("Pareto Frontier: NMI vs Reward")
    ax.legend()
    plt.tight_layout()

    plot_filename = dump_path + f"td_pareto_frontier_{date_time_str}.png"
    plt.savefig(plot_filename, dpi=300)
    print(f"Pareto plot saved to {plot_filename}")

    plt.show()

    print("\nPareto Frontier Parameter Settings:")
    for i in pareto_front:
        print(f"Params: {results[i]['params']}, "
              f"NMI: {results[i]['mean_final_nmi']:.4f} ± {results[i]['std_final_nmi']:.4f}, "
              f"Reward: {results[i]['mean_reward']:.4f} ± {results[i]['std_reward']:.4f}")

    return results, best_params


### Optimization

In [ ]:
# Example usage
param_ranges = {
    # 'learning_rate': (0.001, 0.25),
    "exploration_rate": (0.01, 1),
    "exploration_decay": (0.1, 0.9999999),
    # 'min_exploration_rate': (0.0, 0.05),
    'gamma': (0.1, 1),
    "choice": Categorical(["egreedy", "softmax", "ucb"])
}

td_bayes_results, td_best_params = bayesian_td_parameter_search(
    param_ranges,
    n_calls=200,
    n_trials=100,
    n_episodes=10000,
    n_initial_points=100,
    n_jobs=-1,
    dump_path=dump_path)


# Complex Model

- World States: Three binary variables X, Y, Z
- agents_observed_variables = {0:[0,1],1:[1,2]}
- n_features = 3 #parameters['n_features']
- n_signaling_actions = 4 #parameters['n_signaling_actions']
- n_final_actions = 8 #parameters['n_final_actions']
- Random Games (possibly non-canonical)

## Q-Learning

### Function

In [ ]:
def bayesian_q_parameter_search_complex(
    param_ranges,
    n_calls=30,
    n_trials=10,
    n_episodes=5000,
    base_seed=42,
    n_initial_points=100,
    n_jobs=-1,
    dump_path=dump_path
):
    available_cores = multiprocessing.cpu_count()
    print(f"Available CPU cores: {available_cores}")
    if n_jobs == -1:
        n_jobs = available_cores
    else:
        print(f"Using {n_jobs} cores for parallel execution")

    # Set up named parameter space
    space = []
    for k, v in param_ranges.items():
        if isinstance(v, Categorical):
            space.append(Categorical(v.categories, name=k))
        elif isinstance(v, Real):
            space.append(Real(v.low, v.high, prior=v.prior, name=k))
        elif isinstance(v, tuple):
            space.append(Real(*v, name=k))
        else:
            raise ValueError(f"Unsupported parameter type for '{k}': {v}")

    optimizer = Optimizer(
        dimensions=space,
        base_estimator="GP",
        acq_func="EI",
        random_state=base_seed,
    )

    results = []
    all_params = []
    all_nmis = []
    seen_points = set()

    def point_to_hashable(x):
        return tuple(str(xi) for xi in x)

    def single_trial(params_dict, trial_seed):
        np.random.seed(trial_seed)
        random.seed(trial_seed)

        G = nx.DiGraph()
        G.add_nodes_from([0, 1])
        G.add_edges_from([(0, 1), (1, 0)])

        n_agents = 2
        n_features = 3
        # n_signaling_actions = 4
        # n_final_actions = 8
        n_signaling_actions = np.random.randint(2, 10)
        n_final_actions = np.random.randint(2, 10)
        agents_observed_variables = {0: [0, 1], 1: [1, 2]}

        game = {
            i: create_random_game(n_features, n_final_actions)
            for i in range(n_agents)
        }

        env = NetMultiAgentEnv(
            n_agents=n_agents,
            n_features=n_features,
            n_signaling_actions=n_signaling_actions,
            n_final_actions=n_final_actions,
            full_information=False,
            game_dicts=game,
            observed_variables=agents_observed_variables,
            agent_type=QLearningAgent,
            initialize=False,
            graph=G
        )

        env.agents = [
            QLearningAgent(
                n_signaling_actions=n_signaling_actions,
                n_final_actions=n_final_actions,
                exploration_rate=params_dict['exploration_rate'],
                exploration_decay=params_dict['exploration_decay'],
                min_exploration_rate=1e-10,
                choice=str(params_dict["choice"])
            ) for _ in range(n_agents)
        ]

        _, rewards_history, signal_information_history, _, _ = simulation_function(
            n_agents=n_agents,
            n_features=n_features,
            n_signaling_actions=n_signaling_actions,
            n_final_actions=n_final_actions,
            n_episodes=n_episodes,
            with_signals=True,
            plot=False,
            env=env,
            verbose=False
        )

        final_nmi = [
            np.mean(agent_nmi[-n_episodes // 10:]) if len(agent_nmi) >= n_episodes // 10 else 0.0
            for agent_nmi in signal_information_history
        ]
        final_rewards = [
            np.mean(agent_rewards[-n_episodes // 10:]) if len(agent_rewards) >= n_episodes // 10 else 0.0
            for agent_rewards in rewards_history
        ]

        return np.mean(final_nmi), np.std(final_nmi), np.mean(final_rewards), np.std(final_rewards)

    def evaluate_params(params_dict, seed):
        seeds = [seed + i * 1000 for i in range(n_trials)]
        results = Parallel(n_jobs=n_jobs)(delayed(single_trial)(params_dict, s) for s in seeds)
        nmis, nmi_stds, rewards, reward_stds = zip(*results)
        return np.mean(nmis), np.std(nmis), np.mean(rewards), np.std(rewards)

    for _ in tqdm(range(n_initial_points), desc="Initial Exploration"):
        x = optimizer.ask()
        point_hash = point_to_hashable(x)
        if point_hash in seen_points:
            continue
        seen_points.add(point_hash)

        params_dict = dict(zip(param_ranges.keys(), x))
        seed = base_seed + len(results)
        mean_nmi, std_nmi, mean_reward, std_reward = evaluate_params(params_dict, seed)

        optimizer.tell(x, -mean_nmi)
        results.append({
            "params": params_dict,
            "mean_final_nmi": mean_nmi,
            "std_final_nmi": std_nmi,
            "mean_reward": mean_reward,
            "std_reward": std_reward
        })
        all_params.append(x)
        all_nmis.append(mean_nmi)

    for _ in tqdm(range(n_calls - n_initial_points), desc="Bayesian Optimization"):
        x = optimizer.ask()
        point_hash = point_to_hashable(x)
        if point_hash in seen_points:
            continue
        seen_points.add(point_hash)

        params_dict = dict(zip(param_ranges.keys(), x))
        seed = base_seed + len(results)
        mean_nmi, std_nmi, mean_reward, std_reward = evaluate_params(params_dict, seed)

        optimizer.tell(x, -mean_nmi)
        results.append({
            "params": params_dict,
            "mean_final_nmi": mean_nmi,
            "std_final_nmi": std_nmi,
            "mean_reward": mean_reward,
            "std_reward": std_reward
        })
        all_params.append(x)
        all_nmis.append(mean_nmi)

    df = pd.DataFrame([
        {**r['params'],
         'mean_final_nmi': r['mean_final_nmi'],
         'std_final_nmi': r['std_final_nmi'],
         'mean_reward': r['mean_reward'],
         'std_reward': r['std_reward']} for r in results
    ])

    now = datetime.now()
    date_time_str = now.strftime("%Y-%m-%d_%H-%M-%S")
    save_path = dump_path + f"q_bayes_nmi_results_complex_{n_trials}_{date_time_str}.csv"
    df.to_csv(save_path, index=False)
    print(f"Results saved to {save_path}")

    best_idx = np.argmax(all_nmis)
    best_params = dict(zip(param_ranges.keys(), all_params[best_idx]))
    print(f"Best Parameters: {best_params} with NMI = {all_nmis[best_idx]:.4f}, Reward = {results[best_idx]['mean_reward']:.4f}")

    nmis = df['mean_final_nmi']
    rewards = df['mean_reward']
    nmi_errs = df['std_final_nmi'] / np.sqrt(n_trials)
    reward_errs = df['std_reward'] / np.sqrt(n_trials)

    sorted_indices = np.argsort(-nmis)
    pareto_front = []
    max_y = -float('inf')
    for i in sorted_indices:
        x, y = nmis[i], rewards[i]
        if y > max_y:
            pareto_front.append(i)
            max_y = y

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.set_axisbelow(True)
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
    ax.scatter(nmis, rewards, alpha=0.2, color='gray', label='All Trials')

    pareto_x = nmis.iloc[pareto_front]
    pareto_y = rewards.iloc[pareto_front]
    pareto_xerr = nmi_errs.iloc[pareto_front]
    pareto_yerr = reward_errs.iloc[pareto_front]

    ax.errorbar(pareto_x, pareto_y, xerr=pareto_xerr, yerr=pareto_yerr,
                fmt='o', color='red', ecolor='darkred', capsize=4, label='Pareto Frontier')
    ax.plot(pareto_x, pareto_y, color='darkred', linewidth=2)

    ax.set_xlabel("Mean Final NMI")
    ax.set_ylabel("Mean Reward")
    ax.set_title("Pareto Frontier: NMI vs Reward")
    ax.legend()
    plt.tight_layout()

    plot_filename = dump_path + f"q_pareto_frontier_complex_randomized_{date_time_str}.png"
    plt.savefig(plot_filename, dpi=300)
    print(f"Pareto plot saved to {plot_filename}")
    plt.show()

    print("\nPareto Frontier Parameter Settings:")
    for i in pareto_front:
        print(f"Params: {results[i]['params']}, "
              f"NMI: {results[i]['mean_final_nmi']:.4f} ± {results[i]['std_final_nmi']:.4f}, "
              f"Reward: {results[i]['mean_reward']:.4f} ± {results[i]['std_reward']:.4f}")

    return results, best_params


### Optimization

In [ ]:
# Example usage
param_ranges = {
    "exploration_rate": (0.1, 0.9999999),
    "exploration_decay": (0.1, 0.9999999),
    "choice": Categorical(["egreedy", "softmax", "ucb"])
}

q_bayes_results, q_best_params = bayesian_q_parameter_search_complex(
    param_ranges,
    n_calls=200,
    n_trials=100,
    n_episodes=10000,
    n_initial_points=100,
    n_jobs=-1,
    dump_path=dump_path)

## TD-Learning

### Function

In [ ]:
def bayesian_td_parameter_search_complex(
    param_ranges,
    n_calls=30,
    n_trials=10,
    n_episodes=5000,
    base_seed=42,
    n_initial_points=10,
    n_jobs=-1,
    dump_path=dump_path
):
    available_cores = multiprocessing.cpu_count()
    if n_jobs == -1:
        n_jobs = available_cores

    # Remove 'learning_rate' if fixed
    param_ranges_wo_lr = {k: v for k, v in param_ranges.items() if k != 'learning_rate'}

    # Construct parameter space with correct naming
    space = []
    for k, v in param_ranges_wo_lr.items():
        if isinstance(v, Categorical):
            space.append(Categorical(v.categories, name=k))
        elif isinstance(v, Real):
            space.append(Real(v.low, v.high, prior=v.prior, name=k))
        elif isinstance(v, tuple):
            if isinstance(v[0], float):
                space.append(Real(*v, name=k))
            elif isinstance(v[0], int):
                space.append(Integer(*v, name=k))
            else:
                raise ValueError(f"Unsupported tuple value type for key '{k}': {type(v[0])}")
        else:
            raise ValueError(f"Unsupported parameter type for key '{k}': {type(v)}")

    optimizer = Optimizer(
        dimensions=space,
        base_estimator="GP",
        acq_func="EI",
        random_state=base_seed,
    )

    results = []
    all_params = []
    all_nmis = []

    def single_trial(params_dict, trial_seed):
        np.random.seed(trial_seed)
        random.seed(trial_seed)

        G = nx.DiGraph()
        G.add_nodes_from([0, 1])
        G.add_edges_from([(0, 1), (1, 0)])

        n_agents = 2
        n_features = 3
        # n_signaling_actions = 4
        # n_final_actions = 8
        n_signaling_actions = np.random.randint(2, 10)
        n_final_actions = np.random.randint(2, 10)
        agents_observed_variables = {0: [0, 1], 1: [1, 2]}

        game = {
            i: create_random_game(n_features, n_final_actions)
            for i in range(n_agents)
        }

        env = TempNetMultiAgentEnv(
            n_agents=n_agents,
            n_features=n_features,
            n_signaling_actions=n_signaling_actions,
            n_final_actions=n_final_actions,
            full_information=False,
            game_dicts=game,
            observed_variables=agents_observed_variables,
            agent_type=TDLearningAgent,
            graph=G
        )

        env.agents = [
            TDLearningAgent(
                n_actions=env.max_actions,
                learning_rate=0.1,
                exploration_rate=params_dict['exploration_rate'],
                exploration_decay=params_dict['exploration_decay'],
                min_exploration_rate=1e-10,
                gamma=params_dict['gamma'],
                choice=str(params_dict["choice"])
            ) for _ in range(n_agents)
        ]

        _, rewards_history, signal_information_history, _, _ = temp_simulation_function(
            n_agents=n_agents,
            n_features=n_features,
            n_signaling_actions=n_signaling_actions,
            n_final_actions=n_final_actions,
            n_episodes=n_episodes,
            with_signals=True,
            plot=False,
            env=env,
            verbose=False
        )

        final_nmi = [
            np.mean(agent_nmi[-n_episodes // 10:]) if len(agent_nmi) >= n_episodes // 10 else 0.0
            for agent_nmi in signal_information_history
        ]
        final_rewards = [
            np.mean(agent_rewards[-n_episodes // 10:]) if len(agent_rewards) >= n_episodes // 10 else 0.0
            for agent_rewards in rewards_history
        ]

        return np.mean(final_nmi), np.std(final_nmi), np.mean(final_rewards), np.std(final_rewards)

    def evaluate_params(params_dict, seed):
        seeds = [seed + i * 1000 for i in range(n_trials)]
        results = Parallel(n_jobs=n_jobs)(delayed(single_trial)(params_dict, s) for s in seeds)
        nmis, nmi_stds, rewards, reward_stds = zip(*results)
        return np.mean(nmis), np.std(nmis), np.mean(rewards), np.std(rewards)

    for _ in tqdm(range(n_initial_points), desc="Initial Exploration"):
        x = optimizer.ask()
        params_dict = dict(zip(param_ranges_wo_lr.keys(), x))
        seed = base_seed + len(results)
        mean_nmi, std_nmi, mean_reward, std_reward = evaluate_params(params_dict, seed)
        optimizer.tell(x, -mean_nmi)
        results.append({
            "params": params_dict,
            "mean_final_nmi": mean_nmi,
            "std_final_nmi": std_nmi,
            "mean_reward": mean_reward,
            "std_reward": std_reward
        })
        all_params.append(x)
        all_nmis.append(mean_nmi)

    for _ in tqdm(range(n_calls - n_initial_points), desc="Bayesian Optimization"):
        x = optimizer.ask()
        params_dict = dict(zip(param_ranges_wo_lr.keys(), x))
        seed = base_seed + len(results)
        mean_nmi, std_nmi, mean_reward, std_reward = evaluate_params(params_dict, seed)
        optimizer.tell(x, -mean_nmi)
        results.append({
            "params": params_dict,
            "mean_final_nmi": mean_nmi,
            "std_final_nmi": std_nmi,
            "mean_reward": mean_reward,
            "std_reward": std_reward
        })
        all_params.append(x)
        all_nmis.append(mean_nmi)

    df = pd.DataFrame([{
        **r['params'],
        'mean_final_nmi': r['mean_final_nmi'],
        'std_final_nmi': r['std_final_nmi'],
        'mean_reward': r['mean_reward'],
        'std_reward': r['std_reward']
    } for r in results])

    now = datetime.now()
    date_time_str = now.strftime("%Y-%m-%d_%H-%M-%S")
    save_path = dump_path + f"td_bayes_nmi_results_complex_randomized_{date_time_str}.csv"
    df.to_csv(save_path, index=False)
    print(f"Results saved to {save_path}")

    best_idx = np.argmax(all_nmis)
    best_params = dict(zip(param_ranges_wo_lr.keys(), all_params[best_idx]))
    print(f"Best Parameters: {best_params} with NMI = {all_nmis[best_idx]:.4f}, Reward = {results[best_idx]['mean_reward']:.4f}")

    nmis = df['mean_final_nmi']
    rewards = df['mean_reward']
    nmi_errs = df['std_final_nmi'] / np.sqrt(n_trials)
    reward_errs = df['std_reward'] / np.sqrt(n_trials)

    sorted_indices = np.argsort(-nmis)
    pareto_front = []
    max_y = -float('inf')
    for i in sorted_indices:
        x, y = nmis[i], rewards[i]
        if y > max_y:
            pareto_front.append(i)
            max_y = y

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.set_axisbelow(True)
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
    ax.scatter(nmis, rewards, alpha=0.2, color='gray', label='All Trials')

    pareto_x = nmis.iloc[pareto_front]
    pareto_y = rewards.iloc[pareto_front]
    pareto_xerr = nmi_errs.iloc[pareto_front]
    pareto_yerr = reward_errs.iloc[pareto_front]

    ax.errorbar(pareto_x, pareto_y, xerr=pareto_xerr, yerr=pareto_yerr,
                fmt='o', color='red', ecolor='darkred', capsize=4, label='Pareto Frontier')
    ax.plot(pareto_x, pareto_y, color='darkred', linewidth=2)

    ax.set_xlabel("Mean Final NMI")
    ax.set_ylabel("Mean Reward")
    ax.set_title("Pareto Frontier: NMI vs Reward")
    ax.legend()
    plt.tight_layout()

    plot_filename = dump_path + f"td_pareto_frontier_complex_{date_time_str}.png"
    plt.savefig(plot_filename, dpi=300)
    print(f"Pareto plot saved to {plot_filename}")
    plt.show()

    print("\nPareto Frontier Parameter Settings:")
    for i in pareto_front:
        print(f"Params: {results[i]['params']}, "
              f"NMI: {results[i]['mean_final_nmi']:.4f} ± {results[i]['std_final_nmi']:.4f}, "
              f"Reward: {results[i]['mean_reward']:.4f} ± {results[i]['std_reward']:.4f}")

    return results, best_params


### Optimization

In [ ]:
# Example usage
param_ranges = {
    # 'learning_rate': (0.001, 0.25),
    'exploration_rate': (0.1, 1),
    'exploration_decay': (0.1, 0.9999999),
    # 'min_exploration_rate': (0.000001, 0.05),
    'gamma': (0.1, 1),
    "choice": Categorical(["egreedy", "softmax", "ucb"])
}

td_bayes_results, td_best_params = bayesian_td_parameter_search_complex(
    param_ranges,
    n_calls=200,
    n_trials=100,
    n_episodes=10000,
    n_initial_points=100,
    n_jobs=-1,
    dump_path=dump_path)